In [14]:
import pandas as pd
import os
from datetime import datetime
from datetime import timedelta

# Load the dataset
df_appearances = pd.read_csv("clean_data/j_clean_appearances.csv")

# Show first rows
df_appearances.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,6646,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [15]:
#earliest and latest date in the dataset
print(df_appearances['date'].min())
print(df_appearances['date'].max())

df_appearances['date'] = pd.to_datetime(df_appearances['date'])

#season label function
def get_season_label(date):
    year = date.year
    if date.month >= 7:
        return f"{str(year)[2:]}/{str(year + 1)[2:]}"
    else:
        return f"{str(year - 1)[2:]}/{str(year)[2:]}"

#insert season label column
df_appearances['season'] = df_appearances['date'].apply(get_season_label)

first_game_dates = (
    df_appearances
    .groupby(['player_id', 'season', 'player_club_id'])['date']
    .transform('min')  # transform so it aligns with original DataFrame
)

df_appearances['first_game_date'] = first_game_dates

df_appearances.head()

2012-07-03
2025-04-10


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played,season,first_game_date
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90,12/13,2012-07-03
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90,12/13,2012-07-05
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45,12/13,2012-07-05
3,2234418_73333,2234418,73333,1274,6646,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90,12/13,2012-07-05
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90,12/13,2012-07-05


In [16]:
# rename columns for merge
df_appearances = df_appearances.rename(columns={'player_club_id': 'club_id'})

In [17]:
#grouping the data
df_appearances = df_appearances.groupby(['player_id', 'player_name', 'season', 'first_game_date', 'club_id'])[['minutes_played', 'goals', 'assists', 'yellow_cards', 'red_cards']].sum().reset_index()

In [18]:
# Create a column for season end year as integer
df_appearances['season_end_year'] = df_appearances['season'].apply(lambda x: int('20' + x.split('/')[1]))

# Create default last_game_date: 30 of June
df_appearances['last_game_date'] = pd.to_datetime(df_appearances['season_end_year'].astype(str) + '-06-30')

# Sort by player_id, season, and first_game_date so clubs in the same season are ordered by first game
df_appearances = df_appearances.sort_values(by=['player_id', 'season', 'first_game_date']).reset_index(drop=True)

# Shift the first_game_date column grouped by player and season, to get next club's first_game_date
df_appearances['next_club_first_game'] = df_appearances.groupby(['player_id', 'season'])['first_game_date'].shift(-1)

# For rows where next_club_first_game exists, set last_game_date to day before next club's first_game_date
mask = df_appearances['next_club_first_game'].notna()
df_appearances.loc[mask, 'last_game_date'] = df_appearances.loc[mask, 'next_club_first_game'] - timedelta(days=1)

# Drop helper columns if you want
df_appearances = df_appearances.drop(columns=['season_end_year', 'next_club_first_game'])

# Now df_appearances has 'last_game_date' adjusted for multiple clubs in a season
df_appearances.head(20)

,player_id,player_name,season,first_game_date,club_id,minutes_played,goals,assists,yellow_cards,red_cards,last_game_date
0,10,Miroslav Klose,12/13,2012-08-23,398,2585,16,3,8,0,2013-06-30
1,10,Miroslav Klose,13/14,2013-08-18,398,2220,8,5,2,0,2014-06-30
2,10,Miroslav Klose,14/15,2014-08-24,398,2289,16,9,6,0,2015-06-30
3,10,Miroslav Klose,15/16,2015-08-08,398,1714,8,8,3,0,2016-06-30
4,26,Roman Weidenfeller,12/13,2012-08-12,16,4401,0,0,2,1,2013-06-30
5,26,Roman Weidenfeller,13/14,2013-07-27,16,3855,0,0,1,1,2014-06-30
6,26,Roman Weidenfeller,14/15,2014-08-29,16,2880,0,0,0,0,2015-06-30
7,26,Roman Weidenfeller,15/16,2015-08-06,16,1260,0,0,1,0,2016-06-30
8,26,Roman Weidenfeller,16/17,2016-08-22,16,1020,0,0,0,0,2017-06-30
9,26,Roman Weidenfeller,17/18,2017-11-21,16,92,0,0,0,0,2018-06-30


In [19]:
import os

# create the directory
os.makedirs("to_merge_data", exist_ok=True)

In [20]:
df_appearances.to_csv("to_merge_data/j2_appearances.csv", index=False)